In [0]:
from pyspark.sql import functions as F

bronze_read_df = spark.table(
    "workspace.first_project.orders_bronze"
)

typed_df = (
    bronze_read_df
    .withColumn("region", F.initcap(F.trim(F.col("region"))))
    .withColumn("amount", F.expr("try_cast(amount AS DOUBLE)"))
    .withColumn("order_date", F.expr("try_cast(order_date AS DATE)"))
)

is_valid = (
    F.col("amount").isNotNull()
    & (F.col("amount")>= 0)
    & F.col("order_date").isNotNull()
)
clean_df = typed_df.filter(is_valid).dropDuplicates(["order_id"])
rejected_df = typed_df.filter(~is_valid)

clean_df.write.mode("overwrite").saveAsTable(
    "workspace.first_project.orders_silver"
)
rejected_df.write.mode("overwrite").saveAsTable(
    "workspace.first_project.orders_rejected"
)

print("kept:", clean_df.count(), "rejected:", rejected_df.count())

kept: 95 rejected: 5
